In [1]:
import pandas as pd
import h5py
import matplotlib.pyplot as plt

In [2]:
import numpy as np
import cv2
from pathlib import Path
from scipy.optimize import linear_sum_assignment


def discover_segments(data_dir, prefix="0066"):
    """Find matching fov.h5 / calib.mov pairs sorted by segment index."""
    data_dir = Path(data_dir)
    pairs = []
    for h5_path in sorted(data_dir.glob(f"{prefix}_*_fov.h5")):
        segment = h5_path.stem.split("_")[1]
        mov_path = data_dir / f"{prefix}_{segment}-calib.mov"
        if not mov_path.exists():
            raise FileNotFoundError(f"Missing video for {h5_path.name}: {mov_path}")
        pairs.append((h5_path, mov_path))
    if not pairs:
        raise FileNotFoundError(f"No segments found in {data_dir} for prefix {prefix}")
    return pairs


def _load_segment_fields(h5_path):
    """Load one segment's fields group into a dict."""
    import h5py

    fields = {}
    with h5py.File(h5_path, "r") as f:
        for key in f["fields"].keys():
            val = f["fields"][key][()]
            if isinstance(val, bytes):
                fields[key] = val.decode()
            else:
                fields[key] = np.array(val)
    return fields


def _fish_row_permutation(pos_ref, pos_next):
    """Match fish rows across a segment boundary via Hungarian assignment.

    Returns ``perm`` such that ``next_row[perm[i]]`` is the same individual as
    ``ref_row[i]``.
    """
    pos_ref = np.asarray(pos_ref, dtype=float)
    pos_next = np.asarray(pos_next, dtype=float)
    cost = np.linalg.norm(pos_ref[:, None, :] - pos_next[None, :, :], axis=2)
    cost[~np.isfinite(cost)] = 1e9
    _, col_ind = linear_sum_assignment(cost)
    return col_ind


def _permute_fields_fish(fields, perm, num_fish):
    """Reorder the fish axis (axis 0) for all per-fish arrays."""
    permuted = {}
    for key, val in fields.items():
        if isinstance(val, str) or np.isscalar(val):
            permuted[key] = val
            continue
        arr = np.asarray(val)
        if arr.ndim >= 1 and arr.shape[0] == num_fish:
            permuted[key] = arr[perm]
        else:
            permuted[key] = val
    return permuted


def _merge_aligned_segments(aligned):
    """Concatenate time-aligned segment dicts along the frame axis."""
    keys = list(aligned[0].keys())
    num_fish = np.asarray(aligned[0]["x"]).shape[0]
    fields = {}

    for key in keys:
        val0 = aligned[0][key]
        if isinstance(val0, str) or np.isscalar(val0):
            fields[key] = val0
            continue
        arr0 = np.asarray(val0)
        if arr0.ndim >= 2 and arr0.shape[0] == num_fish:
            fields[key] = np.concatenate([np.asarray(seg[key]) for seg in aligned], axis=1)
        else:
            fields[key] = val0

    return fields


def concat_fields(h5_paths, reidentify=True):
    """Concatenate h5 'fields' groups along the frame axis.

    When ``reidentify`` is True (default), fish row indices are permuted at each
    segment boundary so that row *i* refers to the same individual throughout.
    """
    h5_paths = [Path(p) for p in h5_paths]
    if not h5_paths:
        raise ValueError("No h5 paths provided")

    if not reidentify:
        return _concat_fields_naive(h5_paths)

    chunks = [_load_segment_fields(path) for path in h5_paths]
    num_fish = np.asarray(chunks[0]["x"]).shape[0]
    aligned = [chunks[0]]

    print(f"Cross-segment re-ID across {len(chunks)} segments ({num_fish} fish):")
    for i in range(1, len(chunks)):
        prev = aligned[-1]
        nxt = chunks[i]
        pos_end = np.column_stack([prev["x"][:, -1], prev["y"][:, -1]])
        pos_start = np.column_stack([nxt["x"][:, 0], nxt["y"][:, 0]])
        perm = _fish_row_permutation(pos_end, pos_start)
        match_err = np.linalg.norm(pos_start[perm] - pos_end, axis=1)
        n_moved = int(np.sum(perm != np.arange(num_fish)))
        print(
            f"  {h5_paths[i - 1].name} -> {h5_paths[i].name}: "
            f"{n_moved}/{num_fish} rows permuted, "
            f"median match {np.median(match_err):.1f}px"
        )
        aligned.append(_permute_fields_fish(nxt, perm, num_fish))

    fields = _merge_aligned_segments(aligned)
    n_frames = fields["x"].shape[1]
    print(f"Concatenated {len(h5_paths)} h5 files -> {n_frames} frames (re-ID applied)")
    return fields


def _concat_fields_naive(h5_paths):
    """Original concatenation without cross-segment re-ID."""
    import h5py

    fields = {}
    with h5py.File(h5_paths[0], "r") as f0:
        keys = list(f0["fields"].keys())

    for key in keys:
        with h5py.File(h5_paths[0], "r") as f0:
            ndim = f0["fields"][key].ndim

        if ndim == 0:
            with h5py.File(h5_paths[0], "r") as f0:
                val = f0["fields"][key][()]
            fields[key] = val.decode() if isinstance(val, bytes) else val
        elif ndim == 1:
            with h5py.File(h5_paths[0], "r") as f0:
                fields[key] = np.array(f0["fields"][key])
        else:
            chunks = [np.array(h5py.File(path, "r")["fields"][key]) for path in h5_paths]
            fields[key] = np.concatenate(chunks, axis=1)

    n_frames = fields["x"].shape[1]
    print(f"Concatenated {len(h5_paths)} h5 files -> {n_frames} frames (no re-ID)")
    return fields


def extract_loc_vel_table(fields):
    """Build frame-wise table: x, y, vx, vy for each fish."""
    x = np.asarray(fields["x"], dtype=float)
    y = np.asarray(fields["y"], dtype=float)
    num_fish, num_frames = x.shape

    vx = np.empty_like(x)
    vy = np.empty_like(y)
    vx[:, :-1] = np.diff(x, axis=1)
    vy[:, :-1] = np.diff(y, axis=1)
    vx[:, -1] = np.nan
    vy[:, -1] = np.nan

    columns = []
    data_cols = []
    for fish in range(num_fish):
        for name, arr in (
            (f"fish{fish}_x", x[fish]),
            (f"fish{fish}_y", y[fish]),
            (f"fish{fish}_vx", vx[fish]),
            (f"fish{fish}_vy", vy[fish]),
        ):
            columns.append(name)
            data_cols.append(arr)

    table = np.column_stack(data_cols)
    return table, columns


def save_loc_vel_h5(fields, output_path):
    """Save tabular position/velocity data as h5 and csv."""
    import h5py
    import pandas as pd

    table, columns = extract_loc_vel_table(fields)
    output_path = Path(output_path)
    csv_path = output_path.with_suffix(".csv")

    with h5py.File(output_path, "w") as f:
        f.create_dataset("data", data=table, compression="gzip")
        f.create_dataset(
            "columns",
            data=np.array(columns, dtype=h5py.string_dtype(encoding="utf-8")),
        )
        f.attrs["num_fish"] = table.shape[1] // 4
        f.attrs["num_frames"] = table.shape[0]

    pd.DataFrame(table, columns=columns).to_csv(csv_path, index_label="frame")

    print(f"Saved loc/vel table {table.shape} to {output_path}")
    print(f"Saved loc/vel csv to {csv_path}")
    return output_path, csv_path


def save_fields_h5(fields, output_path):
    """Save a fields dict to an h5 file with the same layout as the source files."""
    import h5py

    output_path = Path(output_path)
    with h5py.File(output_path, "w") as f:
        grp = f.create_group("fields")
        for key, val in fields.items():
            if isinstance(val, str):
                dt = h5py.string_dtype(encoding="utf-8")
                grp.create_dataset(key, data=val, dtype=dt)
            elif isinstance(val, (bytes, np.bytes_)):
                grp.create_dataset(key, data=val.decode(), dtype=h5py.string_dtype(encoding="utf-8"))
            elif np.isscalar(val):
                grp.create_dataset(key, data=val)
            else:
                grp.create_dataset(key, data=np.asarray(val), compression="gzip")

    print(f"Saved concatenated h5 to {output_path}")
    return output_path


def concat_videos(video_paths, output_path):
    """Concatenate videos sequentially into one file."""
    video_paths = [Path(p) for p in video_paths]
    writer = None
    total_frames = 0

    for path in video_paths:
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open video: {path}")

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if writer is None:
                height, width = frame.shape[:2]
                fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
            writer.write(frame)
            total_frames += 1

        cap.release()

    if writer is None:
        raise ValueError("No frames written while concatenating videos")

    writer.release()
    print(f"Concatenated {len(video_paths)} videos -> {total_frames} frames at {output_path}")
    return output_path


def save_heading_video(
    fields,
    video_path,
    output_path="fish_headings.mp4",
    arrow_length=30,
    fish_ids=None,
    start_frame=0,
    end_frame=None,
):
    """Overlay heading arrows per fish on a video and save the result."""
    x = np.asarray(fields["x"])
    y = np.asarray(fields["y"])
    hx = np.asarray(fields["heading_x"])
    hy = np.asarray(fields["heading_y"])

    num_fish, num_frames = x.shape
    if fish_ids is None:
        fish_ids = range(num_fish)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if end_frame is None:
        end_frame = min(num_frames, total_frames)
    else:
        end_frame = min(end_frame, num_frames, total_frames)
    start_frame = max(start_frame, 0)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    for _ in range(start_frame):
        if not cap.grab():
            break

    for t in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break

        for fish in fish_ids:
            if np.isnan(x[fish, t]) or np.isnan(y[fish, t]):
                continue
            px = int(round(x[fish, t]))
            py = int(round(y[fish, t]))
            tipx = px + int(round(arrow_length * hx[fish, t]))
            tipy = py + int(round(arrow_length * hy[fish, t]))
            color = tuple(int(c) for c in np.random.RandomState(fish).randint(0, 255, size=3))
            cv2.arrowedLine(frame, (px, py), (tipx, tipy), color, 2, tipLength=0.2)

        out.write(frame)

    cap.release()
    out.release()
    print(f"Saved heading video with overlays to {output_path}")

In [ ]:
PREFIX = "0084"
DATA_DIR = f"schooling-datasets/30_fish/{PREFIX}"

pairs = discover_segments(DATA_DIR, PREFIX)
h5_paths = [h5 for h5, _ in pairs]
mov_paths = [mov for _, mov in pairs]

print("Segments:")
for h5, mov in pairs:
    print(f"  {h5.name} + {mov.name}")

fields = concat_fields(h5_paths)

concat_h5_path = Path(DATA_DIR) / f"{PREFIX}_fov_concat.h5"
save_fields_h5(fields, concat_h5_path)

loc_vel_path = Path(DATA_DIR) / f"{PREFIX}_loc_vel_data.h5"
save_loc_vel_h5(fields, loc_vel_path)

concat_video_path = Path(DATA_DIR) / f"{PREFIX}_calib_concat.mp4"
concat_videos(mov_paths, concat_video_path)

save_heading_video(
    fields,
    concat_video_path,
    output_path=Path(DATA_DIR) / f"fish_headings_{PREFIX}.mp4",
)

Segments:
  0084_000_fov.h5 + 0084_000-calib.mov
  0084_001_fov.h5 + 0084_001-calib.mov
  0084_002_fov.h5 + 0084_002-calib.mov
  0084_003_fov.h5 + 0084_003-calib.mov
  0084_004_fov.h5 + 0084_004-calib.mov
  0084_005_fov.h5 + 0084_005-calib.mov
  0084_006_fov.h5 + 0084_006-calib.mov
Cross-segment re-ID across 7 segments (30 fish):
  0084_000_fov.h5 -> 0084_001_fov.h5: 25/30 rows permuted, median match 2.8px
  0084_001_fov.h5 -> 0084_002_fov.h5: 28/30 rows permuted, median match 4.5px
  0084_002_fov.h5 -> 0084_003_fov.h5: 29/30 rows permuted, median match 2.4px
  0084_003_fov.h5 -> 0084_004_fov.h5: 30/30 rows permuted, median match 3.1px
  0084_004_fov.h5 -> 0084_005_fov.h5: 27/30 rows permuted, median match 2.2px
  0084_005_fov.h5 -> 0084_006_fov.h5: 29/30 rows permuted, median match 0.9px
Concatenated 7 h5 files -> 24593 frames (re-ID applied)


In [ ]:
# Validate cross-segment continuity (expect ~2-8 px, not hundreds)
import h5py

fps = 29.97
x = np.asarray(fields["x"])
y = np.asarray(fields["y"])
jumps = np.sqrt(np.diff(x, axis=1) ** 2 + np.diff(y, axis=1) ** 2)

segment_lengths = []
for h5 in h5_paths:
    with h5py.File(h5) as f:
        segment_lengths.append(f["fields"]["x"].shape[1])
boundary_frames = np.cumsum(segment_lengths)[:-1] - 1

print("Position jumps at segment boundaries:")
for t in boundary_frames:
    print(f"  {t / fps / 60:.2f} min: median={np.median(jumps[:, t]):.1f}px, max={np.max(jumps[:, t]):.1f}px")